### Anomaly Detection

A different task framing than classification: identify points that do not fit the normal pattern, often with no labeled anomaly examples at all (or very few). Covers Isolation Forest, One-Class SVM, Local Outlier Factor.

## Isolation Forest

#### 0. Core idea

Anomalies are "few and different", they should be easy to isolate with random splits, since they sit far from the dense normal cluster. Build many random trees, each splitting on a random feature at a random threshold, and measure how many splits it takes to isolate each point alone in its own partition. Fewer splits needed to isolate a point (shorter path length) means more anomalous.

Toy setup, 1D: points [1, 2, 3, 4, 5, 50] (50 is an obvious outlier).

#### 1. Worked isolation, by hand

Isolating point 50 (the outlier): first random split, say at threshold 25 (roughly the midpoint of the data range [1,50]). Every normal point (1-5) falls below 25, only 50 falls above. 50 is ALREADY alone after just 1 split. Path length = 1.

Isolating point 3 (buried in the dense cluster): a split at 25 does not separate it from the other 4 normal points, they are all below 25 together. Need splits WITHIN the tight [1,5] range:
```
split 1 (threshold 25): {1,2,3,4,5} stay together, 50 isolated (irrelevant to point 3 now)
split 2 (threshold 2.5, within the [1,5] range): separates {1,2} from {3,4,5}
split 3 (threshold 4.5): separates {3,4} from {5}
split 4 (threshold 3.5): separates {3} from {4}
```
Point 3 needs roughly 4 splits to isolate. Point 50 needed just 1. Average this over many random trees (different random thresholds each time) and the outlier consistently isolates fast, the normal points consistently take longer, that gap IS the anomaly score.

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

X = np.array([1, 2, 3, 4, 5, 50]).reshape(-1, 1)

iso = IsolationForest(n_estimators=100, random_state=42)
iso.fit(X)

scores = iso.decision_function(X)  # higher = more normal, lower/negative = more anomalous
preds = iso.predict(X)             # -1 = anomaly, 1 = normal

for x, s, p in zip(X.flatten(), scores, preds):
    print(f"x={x}: anomaly_score={s:.3f}, prediction={'anomaly' if p == -1 else 'normal'}")

## One-Class SVM

#### 0. Core idea

Same SVM machinery as the classification version (see `classical-ml.ipynb`), but fit on only ONE class, the normal data. Learns a boundary that encloses the normal region as tightly as reasonably possible (controlled by a parameter analogous to soft-margin's C, called nu here, the expected fraction of outliers), anything falling outside that learned boundary at prediction time is flagged anomalous.

#### 1. Practical notes

Same kernel trick as classification SVM applies, an RBF kernel lets the boundary be a non-linear, arbitrarily-shaped enclosure around the normal cluster, not just a hyperplane. Sensitive to nu and the kernel bandwidth (gamma for RBF), too tight a boundary flags normal variation as anomalous, too loose a boundary misses real anomalies. Works best when the normal class is reasonably well-clustered, a normal class that is itself multi-modal (several distinct "normal" sub-patterns) is harder to enclose with one boundary.

In [ ]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(nu=0.15, kernel="rbf", gamma="auto")
ocsvm.fit(X)

preds_svm = ocsvm.predict(X)
for x, p in zip(X.flatten(), preds_svm):
    print(f"x={x}: prediction={'anomaly' if p == -1 else 'normal'}")

## Local Outlier Factor (LOF)

#### 0. Core idea

Density-based, similar spirit to DBSCAN (see `unsupervised.ipynb`), but produces a continuous anomaly SCORE rather than DBSCAN's hard core/border/noise labels. Compares a point's local density to the local density of its neighbors, a point sitting in a notably sparser region than its neighbors gets a high outlier score, even if it is not globally far from everything.

#### 1. Worked by hand, k=2 neighbors

Toy setup: points [1, 2, 3, 10]. LOF's key building block, reachability distance: reach-dist_k(p,o) = max(k-distance(o), dist(p,o)), where k-distance(o) = distance from o to its OWN k-th nearest neighbor. This "max" is deliberate, it prevents a point from getting an artificially tiny reachability distance just because it happens to land extremely close to a neighbor that itself sits in a sparse area.

k-distances (k=2): k-distance(1)=2, k-distance(2)=1, k-distance(3)=2, k-distance(10)=8.

Local reachability density (lrd) = 1 / average reachability distance to a point's own k-nearest neighbors:
```
lrd(1) = 1 / avg(reach-dist(1,2)=1, reach-dist(1,3)=2) = 1/1.5 = 0.667
lrd(2) = 1 / avg(reach-dist(2,1)=2, reach-dist(2,3)=2) = 1/2.0 = 0.500
lrd(3) = 1 / avg(reach-dist(3,2)=1, reach-dist(3,1)=2) = 1/1.5 = 0.667
lrd(10)= 1 / avg(reach-dist(10,3)=7, reach-dist(10,2)=8) = 1/7.5 = 0.133
```
LOF(p) = average, over p's neighbors, of (neighbor's lrd / p's own lrd):
```
LOF(3)  = avg(lrd(2)/lrd(3), lrd(1)/lrd(3)) = avg(0.75, 1.00) = 0.875   ~ 1, normal density
LOF(10) = avg(lrd(3)/lrd(10), lrd(2)/lrd(10)) = avg(5.01, 3.76) = 4.38  >> 1, clear outlier
```
LOF(3)~0.875 means point 3's neighborhood is about as dense as its own neighbors' neighborhoods, unremarkable. LOF(10)=4.38 means point 10's local density is roughly 4x LOWER than its neighbors', a real, sharply-quantified outlier signal, not just a binary flag.

#### 2. Why "local" matters

A single global density threshold (like DBSCAN's eps) can fail when different regions of the data have genuinely different natural densities, a point that would be normal-density in a sparse region might look artificially anomalous under a threshold tuned for a denser region elsewhere in the same dataset. LOF compares each point only against ITS OWN neighborhood's density, not a single global standard, which is what "local" refers to.

#### 3. Practical notes

Good for datasets with clusters of varying density, exactly DBSCAN's weak point. Cost: like KNN and DBSCAN, needs pairwise distance computation, does not scale as well as Isolation Forest to very large datasets. Output is a continuous score (negative_outlier_factor_ in sklearn), useful for ranking "most anomalous first" rather than only a binary flag.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
import numpy as np

X_small = np.array([1, 2, 3, 10]).reshape(-1, 1)

lof_small = LocalOutlierFactor(n_neighbors=2)
lof_small.fit_predict(X_small)
lof_scores = -lof_small.negative_outlier_factor_  # sklearn negates it, flip back to match the hand-worked LOF values

for x, score in zip(X_small.flatten(), lof_scores):
    print(f"x={x}: LOF={score:.3f}")

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=3)
preds_lof = lof.fit_predict(X)
scores_lof = lof.negative_outlier_factor_

for x, s, p in zip(X.flatten(), scores_lof, preds_lof):
    print(f"x={x}: LOF_score={s:.3f}, prediction={'anomaly' if p == -1 else 'normal'}")

#### Comparing the three, when to use which

| | Isolation Forest | One-Class SVM | LOF |
|---|---|---|---|
| Mechanism | random-split path length | learned boundary around normal region | local density ratio vs. neighbors |
| Handles varying density across the dataset | no, single global notion of "easy to isolate" | no, one boundary for the whole normal class | yes, this is exactly what it's built for |
| Scales to large data | best of the three, tree-based, no pairwise distances | moderate, kernel computation cost grows with n | worst, needs pairwise distances like KNN/DBSCAN |
| Output | anomaly score (path-length based) | hard in/out decision (or a score via decision_function) | continuous score, good for ranking |
| Best fit | large datasets, no strong density-variation concerns | normal class is unimodal/well-clustered, moderate size | normal data has multiple sub-clusters of different densities |

Rule of thumb: start with Isolation Forest for a quick, scalable baseline, reach for LOF specifically when you suspect the normal class isn't one uniform density (common in fraud data, "normal" behavior for a high-volume merchant looks nothing like "normal" for a low-volume one), and One-Class SVM when you have a clean, reasonably small, well-clustered normal class and want an explicit learned boundary.

#### Relevance to fraud detection specifically

These methods answer a genuinely different question than the classification notebooks in this series (`classical-ml.ipynb`, `boosting.ipynb`): "is this normal or not", with no labeled fraud examples required at all, versus "which specific typology is this", which needs labeled examples of every typology to train on. Anomaly detection is the right tool when fraud is rare enough, or novel enough, that you do not have (or trust) labeled examples of it yet, a genuinely new scam pattern would not be classified correctly by the fraud-theme-detection project's supervised model (it was only trained on the 10 known typologies), but could still get flagged as anomalous by these methods purely for not looking like normal activity.